# Model Evaluation — Recruiter Bot Action Classification

This notebook evaluates how well the **Main Agent** classifies recruiter turns as
`continue`, `schedule`, or `end` compared to the ground-truth labels in
`sms_conversations.json`.

**Metrics reported:**
- Overall Accuracy
- Per-class Precision, Recall, F1
- Confusion Matrix

---
**Note:** Each prediction calls the OpenAI API once (single LLM call, no advisors).
With 15 conversations and ~3 labeled turns each, expect ~45 API calls total.

## 1. Setup

In [ ]:
import sys
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from dotenv import load_dotenv

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

print("Setup complete.")
print(f"Project root: {PROJECT_ROOT}")

## 2. Load Labeled Dataset

In [ ]:
CONVERSATIONS_PATH = os.path.join(PROJECT_ROOT, "sms_conversations.json")

with open(CONVERSATIONS_PATH, "r", encoding="utf-8") as f:
    conversations = json.load(f)

print(f"Loaded {len(conversations)} conversations.")


def build_eval_samples(conversations: list) -> list[dict]:
    """
    For every recruiter turn that has a label, build:
      - history_text: the full conversation up to (and including) that turn
      - true_label:   the ground-truth action
    """
    samples = []
    for conv in conversations:
        turns = conv["turns"]
        for idx, turn in enumerate(turns):
            if turn["speaker"] != "recruiter" or turn["label"] is None:
                continue
            # Build history text
            lines = []
            for t in turns[: idx + 1]:
                role = "Recruiter" if t["speaker"] == "recruiter" else "Candidate"
                lines.append(f"{role}: {t['text']}")
            samples.append({
                "conversation_id": conv["conversation_id"],
                "turn_id": turn["turn_id"],
                "history_text": "\n".join(lines),
                "true_label": turn["label"],
            })
    return samples


samples = build_eval_samples(conversations)
df = pd.DataFrame(samples)

print(f"\nTotal labeled turns: {len(df)}")
print("\nLabel distribution:")
print(df["true_label"].value_counts())

## 3. Run Predictions

We use `MainAgent.decide_action_only()` — a lightweight method that makes a single
LLM call per turn (no advisor agents) to classify the action.

In [ ]:
from app.modules.main_agent.main_agent import MainAgent
from app.modules.exit_advisor.exit_advisor import ExitAdvisor
from app.modules.scheduling_advisor.scheduling_advisor import SchedulingAdvisor
from app.modules.info_advisor.info_advisor import InfoAdvisor

# For evaluation we only need the main agent's decision logic
agent = MainAgent(
    exit_advisor=ExitAdvisor(),
    scheduling_advisor=SchedulingAdvisor(),
    info_advisor=InfoAdvisor(),
)

print(f"Running predictions on {len(df)} turns… (one API call each)")

predictions = []
for i, row in df.iterrows():
    pred = agent.decide_action_only(row["history_text"])
    predictions.append(pred)
    print(f"  [{i+1:02d}/{len(df)}] conv={row['conversation_id']} turn={row['turn_id']} "
          f"true={row['true_label']:8s} pred={pred}")

df["predicted_label"] = predictions
print("\nDone.")

## 4. Results

In [ ]:
y_true = df["true_label"]
y_pred = df["predicted_label"]

accuracy = accuracy_score(y_true, y_pred)
print(f"Overall Accuracy: {accuracy:.1%}\n")
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["continue", "end", "schedule"]))

## 5. Confusion Matrix

In [ ]:
labels = ["continue", "schedule", "end"]
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5,
    ax=ax,
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title(f"Confusion Matrix  (Accuracy = {accuracy:.1%})", fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "tests", "confusion_matrix.png"), dpi=150)
plt.show()
print("Plot saved to tests/confusion_matrix.png")

## 6. Detailed Results Table

In [ ]:
df["correct"] = df["true_label"] == df["predicted_label"]

display_cols = ["conversation_id", "turn_id", "true_label", "predicted_label", "correct"]
styled = (
    df[display_cols]
    .style
    .applymap(lambda v: "background-color: #d4edda" if v is True else
                        ("background-color: #f8d7da" if v is False else ""),
              subset=["correct"])
    .set_caption("Green = correct prediction | Red = mismatch")
)
display(styled)

## 7. Error Analysis

In [ ]:
errors = df[df["correct"] == False][["conversation_id", "turn_id", "true_label", "predicted_label", "history_text"]]

if errors.empty:
    print("No errors — perfect score!")
else:
    print(f"{len(errors)} misclassified turns:\n")
    for _, row in errors.iterrows():
        print(f"Conv {row['conversation_id']} Turn {row['turn_id']}: "
              f"true={row['true_label']} | pred={row['predicted_label']}")
        print(f"  Last line: {row['history_text'].splitlines()[-1]}")
        print()